<!-- NB_DOC_INTRO_V1 -->
# Inference causale

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Correlation ≠ Causation.** Un modele predictif sait dire "qui va churn", mais pas "que se passe-t-il si on offre une promo a ces users ?". Pour cette question (contrefactuelle), il faut **l'inference causale**.

Ce notebook execute les **methodes principales** :
- **Regression adjustment** (statsmodels) — confondants observes
- **Propensity Score Matching / Weighting** — equilibrer les groupes traites/non-traites
- **Difference-in-Differences (DiD)** — donnees panel avant/apres
- **Doubly Robust** — combinaison IPW + regression
- **DoWhy framework** — workflow 4 etapes (Model → Identify → Estimate → Refute)

Concepts cles :
- **ATE** (Average Treatment Effect) = `E[Y(1) - Y(0)]`
- **ATT** (Average Treatment effect on Treated) = `E[Y(1) - Y(0) | T=1]`
- **CATE** (Conditional ATE) = `E[Y(1) - Y(0) | X]` — heterogene
- **Confondant** : variable causant a la fois T et Y → biaise l'estimation naive
- **DAG** : graphe oriente acyclique decrivant les relations causales

> 🎯 **Hierarchie de Pearl** : Association (stats / ML) < Intervention (do-calculus) < Counterfactual (modeles structurels). Inference causale = niveaux 2 et 3.

Versions : `dowhy >= 0.11`, `statsmodels >= 0.14`, `scikit-learn >= 1.4`.

## Plan

1. Setup + generation de donnees synthetiques avec **vraie cause connue**
2. **Biais d'estimation naive** (regression sans ajustement)
3. **Regression adjustment** (statsmodels)
4. **Propensity Score Matching**
5. **Inverse Propensity Weighting (IPW)**
6. **Doubly Robust** (combine IPW + regression)
7. **DiD** (Difference-in-Differences)
8. **DoWhy framework** complet (4 etapes + refute)
9. **Heterogeneous Treatment Effects** (CATE)
10. Pieges et anti-patterns
11. References


## 1. Donnees synthetiques avec vraie cause connue

On simule un cas ou on **connait** le vrai effet causal (`ATE_vrai`) pour pouvoir comparer chaque methode.

**Scenario** : effet d'un programme de formation (T) sur le salaire (Y), avec confondant **education** (X1) qui influence a la fois la probabilite de faire la formation ET le salaire.


In [ ]:
# pip install -q dowhy statsmodels scikit-learn numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor

SEED = 42
np.random.seed(SEED)

# === Generation : vraie cause connue ===
N = 5000

# Confondant : niveau d'education (1-5)
education = np.random.randint(1, 6, N)

# Age (autre confondant)
age = np.random.randint(20, 60, N)

# Probabilite de faire la formation (T) : plus eduque + jeune → plus de chance
logit_T = -1.0 + 0.4 * education - 0.02 * age + np.random.randn(N) * 0.3
prob_T = 1 / (1 + np.exp(-logit_T))
T = (np.random.rand(N) < prob_T).astype(int)

# Salaire (Y) : depend d'education, age, ET formation (le vrai effet causal)
ATE_VRAI = 3000   # la formation augmente le salaire de 3000 € en moyenne
Y = 20000 + 4000 * education + 200 * age + ATE_VRAI * T + np.random.randn(N) * 2000

df = pd.DataFrame({"education": education, "age": age, "T": T, "Y": Y})
print(f"N = {N}")
print(f"Taux de traites : {T.mean():.2%}")
print(f"Salaire moyen traites    : {Y[T==1].mean():.0f}")
print(f"Salaire moyen non-traites: {Y[T==0].mean():.0f}")
print(f"Difference naive         : {Y[T==1].mean() - Y[T==0].mean():.0f}  <- BIAISE")
print(f"Vrai effet causal (ATE)  : {ATE_VRAI}")

**Lecture** : la difference naive (`mean(Y|T=1) - mean(Y|T=0)`) est tres differente de l'ATE vrai. Pourquoi ? Parce que les traites sont **systematiquement plus eduques** (donc auraient gagne plus meme sans la formation).


## 2. Diagnostic du biais : difference naive vs verite


In [ ]:
# Estimateur naif (juste la difference de moyennes)
naive_estimate = Y[T == 1].mean() - Y[T == 0].mean()
print(f"Estimateur naif : {naive_estimate:.0f}")
print(f"Vrai ATE        : {ATE_VRAI}")
print(f"Biais           : {naive_estimate - ATE_VRAI:+.0f}  (~{100*(naive_estimate-ATE_VRAI)/ATE_VRAI:+.0f}%)")

## 3. Regression adjustment

Methode la plus simple : **ajouter les confondants comme features** dans une regression lineaire. Le coef sur T est l'effet causal **conditionnellement** aux confondants.

`Y = α + β·T + γ₁·education + γ₂·age + ε`

→ `β̂` ≈ ATE si tous les confondants sont observes et la relation est lineaire.


In [ ]:
X = df[["T", "education", "age"]]
X = sm.add_constant(X)
model_ols = sm.OLS(df["Y"], X).fit()
print(model_ols.summary())

In [ ]:
ate_ols = model_ols.params["T"]
ci_lo, ci_hi = model_ols.conf_int().loc["T"]
print(f"\nATE (regression adjustment) : {ate_ols:.0f}  [95% CI : {ci_lo:.0f}, {ci_hi:.0f}]")
print(f"Vrai ATE                    : {ATE_VRAI}")
print(f"Biais residuel              : {ate_ols - ATE_VRAI:+.0f}")

**Bien mieux que le naif** (3000 ± ~150 vs 6000+ naif). Mais attention : la regression suppose le **modele bien specifie** (lineaire + confondants tous observes).


## 4. Propensity Score Matching

L'**ePropensity score** = `e(x) = P(T=1 | X=x)`. Idee : apparier chaque traite avec un non-traite ayant un score similaire. Apres matching, comparaison ~ valide.


In [ ]:
# Estimer le propensity score avec LogReg
ps_model = LogisticRegression(max_iter=1000, random_state=SEED).fit(
    df[["education", "age"]], df["T"]
)
df["ps"] = ps_model.predict_proba(df[["education", "age"]])[:, 1]

# Visualiser : distribution des PS dans les 2 groupes
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df.loc[df["T"] == 1, "ps"], bins=30, alpha=0.5, label="Traites", color="C0")
ax.hist(df.loc[df["T"] == 0, "ps"], bins=30, alpha=0.5, label="Non-traites", color="C3")
ax.set(xlabel="Propensity score", ylabel="Frequence",
       title="Distribution des Propensity Scores par groupe")
ax.legend()
plt.show()

In [ ]:
# Matching 1:1 nearest neighbor sur PS
from sklearn.neighbors import NearestNeighbors

treated = df[df["T"] == 1].copy()
control = df[df["T"] == 0].copy()

nn = NearestNeighbors(n_neighbors=1).fit(control[["ps"]].values)
distances, indices = nn.kneighbors(treated[["ps"]].values)

matched_control = control.iloc[indices.flatten()].reset_index(drop=True)
treated = treated.reset_index(drop=True)

# ATT = effet sur les traites (effet moyen pour ceux qui ont recu le traitement)
att_psm = (treated["Y"] - matched_control["Y"]).mean()
print(f"ATT (PSM 1:1)      : {att_psm:.0f}")
print(f"Vrai ATE           : {ATE_VRAI}  (ATT proche d'ATE ici car effet homogene)")

## 5. Inverse Propensity Weighting (IPW)

Ponderer chaque observation par `1/e(x)` pour les traites et `1/(1-e(x))` pour les non-traites → l'echantillon pondere devient comme un RCT.


In [ ]:
# Ponderation
weight = np.where(df["T"] == 1, 1 / df["ps"], 1 / (1 - df["ps"]))

# Clipper les poids extremes (pratique courante : poids dans [0.05, 0.95])
ps_clipped = np.clip(df["ps"], 0.05, 0.95)
weight = np.where(df["T"] == 1, 1 / ps_clipped, 1 / (1 - ps_clipped))

# IPW estimator (Horvitz-Thompson)
ate_ipw = (df["Y"] * df["T"] * weight).sum() / weight[df["T"] == 1].sum() - \
           (df["Y"] * (1 - df["T"]) * weight).sum() / weight[df["T"] == 0].sum()

print(f"ATE (IPW)         : {ate_ipw:.0f}")
print(f"Vrai ATE          : {ATE_VRAI}")
print(f"Min poids         : {weight.min():.2f}")
print(f"Max poids         : {weight.max():.2f}")

## 6. Doubly Robust (AIPW)

Combine **regression** et **IPW** : robuste si AU MOINS UN des deux est correctement specifie. **Recommande en pratique** par les statisticiens modernes (Bang-Robins 2005).

Formula :
```
AIPW = mean(  T*(Y - mu1(X))/e(X) + mu1(X)  )
     - mean( (1-T)*(Y - mu0(X))/(1-e(X)) + mu0(X) )
```
ou `mu_t(X)` = E[Y | T=t, X] (regressions par groupe).


In [ ]:
# Modeles de regression par groupe
m1 = LinearRegression().fit(df.loc[df["T"]==1, ["education","age"]], df.loc[df["T"]==1, "Y"])
m0 = LinearRegression().fit(df.loc[df["T"]==0, ["education","age"]], df.loc[df["T"]==0, "Y"])

mu1 = m1.predict(df[["education", "age"]])
mu0 = m0.predict(df[["education", "age"]])

# AIPW
aipw_term1 = (df["T"] * (df["Y"] - mu1) / ps_clipped + mu1).mean()
aipw_term0 = ((1 - df["T"]) * (df["Y"] - mu0) / (1 - ps_clipped) + mu0).mean()
ate_aipw = aipw_term1 - aipw_term0

print(f"ATE (AIPW)        : {ate_aipw:.0f}")
print(f"Vrai ATE          : {ATE_VRAI}")

## 7. Difference-in-Differences (DiD)

Cas : **donnees panel** avec une intervention qui touche un groupe a un moment donne. On compare la **variation** entre groupes.

`Y_DiD = (Y_traite_apres - Y_traite_avant) - (Y_controle_apres - Y_controle_avant)`

**Hypothese cruciale : parallel trends** (sans l'intervention, les 2 groupes auraient evolue de la meme facon).


In [ ]:
# Simuler un dataset panel
np.random.seed(SEED)
n_units = 200
units = np.arange(n_units)
treated_units = units < 100  # premieres 100 unites sont traitees

records = []
for u in units:
    # Tendance commune + bruit par unite
    base = np.random.randn() * 5
    trend = np.array([0, 2, 4, 6])  # croissance lineaire dans le temps
    if treated_units[u]:
        # Effet du traitement actif a partir de t=2 (intervention)
        treatment_effect = np.array([0, 0, 5, 8])
    else:
        treatment_effect = np.zeros(4)

    for t in range(4):
        y = 10 + base + trend[t] + treatment_effect[t] + np.random.randn() * 1.0
        records.append({"unit": u, "t": t, "y": y,
                       "treated": int(treated_units[u]),
                       "post": int(t >= 2)})

panel = pd.DataFrame(records)
print(panel.head())

In [ ]:
# DiD via OLS avec interaction Post × Treated
# Y = α + β1·Post + β2·Treated + β3·(Post×Treated) + ε
# β3 = effet causal du traitement (DiD)

import statsmodels.formula.api as smf

did_model = smf.ols("y ~ post + treated + post:treated", data=panel).fit()
print(did_model.summary().tables[1])

did_estimate = did_model.params["post:treated"]
print(f"\nDiD estimate : {did_estimate:.2f}  (vrai effet ~ 6.5 = moyenne de [5, 8])")

## 8. DoWhy framework — workflow standardise

DoWhy formalise l'inference causale en 4 etapes :
1. **Model** : declarer le DAG (relations causales supposees)
2. **Identify** : verifier l'identifiabilite (back-door / front-door / IV)
3. **Estimate** : calculer l'effet
4. **Refute** : tests de sensibilite (placebo, random common cause, subset)


In [ ]:
import dowhy
from dowhy import CausalModel

# 1. Model — declarer le DAG
model = CausalModel(
    data=df,
    treatment="T",
    outcome="Y",
    graph='digraph { T -> Y; education -> T; education -> Y; age -> T; age -> Y; }',
)
print(f"DoWhy version : {dowhy.__version__}")

In [ ]:
# 2. Identify — quelles formules d'estimation sont possibles ?
identified = model.identify_effect(proceed_when_unidentifiable=True)
print("Identified estimands :")
print(identified)

In [ ]:
# 3. Estimate — utiliser plusieurs methodes et comparer
methods = [
    "backdoor.linear_regression",
    "backdoor.propensity_score_matching",
    "backdoor.propensity_score_weighting",
]

results = {}
for m in methods:
    est = model.estimate_effect(identified, method_name=m, test_significance=False)
    results[m] = est.value

print("Estimations DoWhy :")
for m, v in results.items():
    print(f"  {m:50s} : {v:.0f}")
print(f"  Vrai ATE                                            : {ATE_VRAI}")

In [ ]:
# 4. Refute — tests de robustesse
estimate = model.estimate_effect(identified, method_name="backdoor.linear_regression")

# 4a. Random common cause : ajouter une feature aleatoire au DAG -> l'estimation devrait pas changer
refute_random = model.refute_estimate(identified, estimate, method_name="random_common_cause")
print(f"Random common cause :")
print(f"  Original : {refute_random.estimated_effect:.0f}")
print(f"  Apres    : {refute_random.new_effect:.0f}  (proche d'original = robuste)")

# 4b. Placebo : remplacer T par du bruit -> l'effet devrait s'annuler
refute_placebo = model.refute_estimate(identified, estimate, method_name="placebo_treatment_refuter")
print(f"\nPlacebo treatment :")
print(f"  Original          : {refute_placebo.estimated_effect:.0f}")
print(f"  Avec placebo      : {refute_placebo.new_effect:.0f}  (proche de 0 = robuste)")

## 9. Heterogeneous Treatment Effects (CATE)

CATE = effet causal **conditionnel** aux features : `τ(x) = E[Y(1) - Y(0) | X=x]`. Permet de cibler : "a qui ce traitement profite le plus ?".

Methodes : **Causal Forests** (Wager & Athey 2018), **DoubleML**, **Meta-learners** (S/T/X/R/DR-learners).


In [ ]:
# Pseudo-code Causal Forest (necessite econml)
# from econml.dml import CausalForestDML
# cf = CausalForestDML(model_y=RandomForestRegressor(),
#                      model_t=LogisticRegression(),
#                      n_estimators=100, random_state=SEED)
# cf.fit(Y=df["Y"], T=df["T"], X=df[["education", "age"]])
# cate = cf.effect(df[["education", "age"]])
# print(f"CATE moyen : {cate.mean():.0f}, std : {cate.std():.0f}")

# Version simplifie : T-learner manuel
m1_full = RandomForestRegressor(n_estimators=50, random_state=SEED).fit(
    df.loc[df["T"]==1, ["education","age"]], df.loc[df["T"]==1, "Y"]
)
m0_full = RandomForestRegressor(n_estimators=50, random_state=SEED).fit(
    df.loc[df["T"]==0, ["education","age"]], df.loc[df["T"]==0, "Y"]
)

# CATE = mu1(x) - mu0(x) pour chaque x
cate = m1_full.predict(df[["education", "age"]]) - m0_full.predict(df[["education", "age"]])
print(f"CATE (T-learner) :")
print(f"  Moyen : {cate.mean():.0f}")
print(f"  Std   : {cate.std():.0f}")
print(f"  Range : [{cate.min():.0f}, {cate.max():.0f}]")

# Visualiser CATE en fonction de education
fig, ax = plt.subplots(figsize=(10, 4))
for edu in sorted(df["education"].unique()):
    mask = df["education"] == edu
    ax.hist(cate[mask], bins=20, alpha=0.5, label=f"Education={edu}")
ax.set(xlabel="CATE estime (€)", ylabel="Frequence", title="CATE par niveau d'education")
ax.axvline(ATE_VRAI, color="k", ls="--", label=f"Vrai ATE = {ATE_VRAI}")
ax.legend()
plt.show()

## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Conditionner sur un **collider** (variable causee par T ET Y) | Cree de la correlation **fictive** → ne pas conditionner |
| Conditionner sur un **mediator** (sur le chemin T→M→Y) | Masque l'effet → ne pas conditionner sauf cas particulier |
| Ignorer les confondants non observes | Sensitivity analysis (E-value) ou IV |
| Confondre ATE / ATT / CATE / LATE | ATE = pop entiere, ATT = sur les traites, CATE = par segment |
| Causal forest sans cross-fitting | Utiliser `DoubleML` |
| Hypothese de **parallel trends** non testee en DiD | Verifier sur les pre-trends |
| Propensity Score = 0 ou 1 | Trimming / overlap requise (positivite) |
| Donner un effet causal a partir d'un modele predictif (SHAP) | SHAP ≠ effet causal (correlationnel) |
| Ne pas faire de **refute** | Toujours tester (placebo, random common cause) |


## 11. References

### Livres et papers fondateurs
- **Pearl & Mackenzie (2018)**. *The Book of Why*. Vulgarisation accessible.
- **Hernan & Robins**. *Causal Inference: What If*. **Gratuit en ligne** : https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/
- **Imbens & Rubin (2015)**. *Causal Inference for Statistics, Social, and Biomedical Sciences*.
- Wager & Athey (2018). *Estimation and Inference of Heterogeneous Treatment Effects using Random Forests*.

### Documentation officielle
- **DoWhy** : https://www.pywhy.org/dowhy/
- **EconML** : https://econml.azurewebsites.net/
- **CausalML** (Uber) : https://causalml.readthedocs.io/
- **CausalNex** (QuantumBlack) : https://causalnex.readthedocs.io/

### Voir aussi (collection)
- [DS_Bayesian.ipynb](./DS_Bayesian.ipynb) — A/B test bayesien (forme simple d'inference causale)
- [ML_Apprentissage_par_Renforcement.ipynb](./ML_Apprentissage_par_Renforcement.ipynb) — bandits = inference causale en ligne
- [ML_Explication_Feature_Importance_Selection.ipynb](./ML_Explication_Feature_Importance_Selection.ipynb) — SHAP ≠ causal
